# Bölüm 4: Rassal Değişkenlerin Dağılımları

**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, istatistikteki temel olasılık dağılımlarını Python ile uygulamalı olarak ele almaktadır:

| Konu | İçerik |
|------|---------|
| 1. Normal Dağılım | Z skoru, yüzdelik dilim, 68-95-99.7 kuralı |
| 2. Geometrik Dağılım | İlk başarıya kadar bekleme süresi |
| 3. Binom Dağılımı | n denemede k başarı olasılığı |
| 4. Negatif Binom Dağılımı | n. denemede k. başarı |
| 5. Poisson Dağılımı | Nadir olayların modellenmesi |

---

In [ ]:
# Gerekli kütüphanelerin yüklenmesi
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.special import comb, factorial
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Renk paleti
MAVI   = '#2E86AB'
TURUNCU = '#E07A5F'
YESIL  = '#3D9970'
MOR    = '#6C5B7B'
GRI    = '#C0C0C0'

print('Kütüphaneler başarıyla yüklendi ✓')

---
## 1. Normal Dağılım

### 1.1 Teori

Normal dağılım **tek tepeli, simetrik, çan biçiminde** bir sürekli olasılık dağılımıdır. İki parametre ile tanımlanır:

$$X \sim N(\mu, \sigma)$$

- $\mu$: Ortalama (dağılımın merkezi)
- $\sigma$: Standart sapma (yayılım)

**Olasılık yoğunluk fonksiyonu:**

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

> 📌 **Önemli:** Pek çok gerçek yaşam değişkeni *yaklaşık* olarak normaldir; hiçbiri tam olarak normal değildir.

In [ ]:
# ── Farklı parametrelerle normal dağılımlar ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    (0,  1,  'N(μ=0, σ=1) — Standart Normal', MAVI),
    (19, 4,  'N(μ=19, σ=4)',                   TURUNCU),
    (0,  1,  'Karşılaştırma: σ=0.5 vs σ=2',   YESIL),
]

# Sol: Standart normal
x = np.linspace(-4, 4, 300)
axes[0].plot(x, stats.norm.pdf(x, 0, 1), color=MAVI, lw=2.5)
axes[0].fill_between(x, stats.norm.pdf(x, 0, 1), alpha=0.15, color=MAVI)
axes[0].set_title('N(μ=0, σ=1) — Standart Normal', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('Yoğunluk')
axes[0].axvline(0, color='red', lw=1.5, ls='--', label='μ=0')
axes[0].legend()

# Orta: Farklı ortalama
x2 = np.linspace(3, 35, 300)
axes[1].plot(x2, stats.norm.pdf(x2, 19, 4), color=TURUNCU, lw=2.5)
axes[1].fill_between(x2, stats.norm.pdf(x2, 19, 4), alpha=0.15, color=TURUNCU)
axes[1].set_title('N(μ=19, σ=4)', fontweight='bold')
axes[1].set_xlabel('x'); axes[1].set_ylabel('Yoğunluk')
axes[1].axvline(19, color='red', lw=1.5, ls='--', label='μ=19')
axes[1].legend()

# Sağ: Farklı standart sapmalar
x3 = np.linspace(-6, 6, 300)
for sigma, color, label in [(0.5, MAVI, 'σ=0.5'), (1, TURUNCU, 'σ=1'), (2, YESIL, 'σ=2')]:
    axes[2].plot(x3, stats.norm.pdf(x3, 0, sigma), color=color, lw=2, label=label)
axes[2].set_title('Farklı σ değerleri (μ=0)', fontweight='bold')
axes[2].set_xlabel('x'); axes[2].set_ylabel('Yoğunluk')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Normal Dağılım: Parametre Etkisi', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('normal_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ σ arttıkça dağılım yayılır; μ değiştikçe dağılım sola/sağa kayar.')

---
### 1.2 Z Skoru ile Standartlaştırma

Farklı ölçeklerdeki dağılımları karşılaştırmak için her gözlemi **standart birime** çeviririz:

$$Z = \frac{\text{gözlem} - \mu}{\sigma}$$

Z skoru, bir değerin ortalamanın kaç standart sapma **üstünde (+)** ya da **altında (−)** olduğunu gösterir.

> 🔑 **Kural:** $|Z| > 2$ olan gözlemler genellikle **alışılmadık** kabul edilir.

In [ ]:
# ── SAT vs ACT Karşılaştırması (Slayt 5-6) ───────────────────────────────────
# SAT: μ=1500, σ=300   →  Pam: 1800
# ACT: μ=21,  σ=5      →  Jim: 24

z_pam = (1800 - 1500) / 300
z_jim = (24   - 21)   / 5

print('=' * 50)
print('SAT vs ACT — Kim daha başarılı?')
print('=' * 50)
print(f"Pam'in SAT puanı : 1800  →  Z = (1800-1500)/300 = {z_pam:.2f}")
print(f"Jim'in ACT puanı : 24    →  Z = (24-21)/5       = {z_jim:.2f}")
print()
kazanan = "Pam" if z_pam > z_jim else "Jim"
print(f'🏆 {kazanan}, akranlarına kıyasla daha yüksek Z skoruna sahip!')

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
renk_bilgi = [('Pam — SAT', 1500, 300, 1800, MAVI, axes[0]),
              ('Jim — ACT', 21,   5,   24,   TURUNCU, axes[1])]

for baslik, mu, sigma, deger, renk, ax in renk_bilgi:
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color=renk, lw=2.5)
    ax.fill_between(x, stats.norm.pdf(x, mu, sigma),
                    where=(x <= deger), alpha=0.2, color=renk)
    ax.axvline(deger, color=renk, lw=2, ls='--')
    ax.axvline(mu, color='gray', lw=1.5, ls=':')
    ax.set_title(f'{baslik}\nZ = {(deger-mu)/sigma:.2f}', fontweight='bold')
    ax.set_xlabel('Puan'); ax.set_ylabel('Yoğunluk')
    z_val = (deger - mu) / sigma
    ax.text(deger, ax.get_ylim()[1]*0.5, f' Z={z_val:.1f}', color=renk, fontweight='bold')

plt.tight_layout()
plt.savefig('z_skoru.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 1.3 Yüzdelik Dilimler

**Yüzdelik dilim**, belirli bir değerin *altında* kalan gözlemlerin yüzdesidir. Grafikte bu, eğrinin **solundaki alan**dır.

Python'da hesaplama:
- Alan (kümülatif olasılık): `scipy.stats.norm.cdf(x, mu, sigma)`  
- Tersine (değer bul): `scipy.stats.norm.ppf(p, mu, sigma)`

In [ ]:
# ── Kalite Kontrol — Heinz Ketçap (Slayt 12-14) ──────────────────────────────
mu_ketchup = 36
sigma_ketchup = 0.11

print('=' * 55)
print('Heinz Ketçap Kalite Kontrol')
print(f'X ~ N(μ={mu_ketchup}, σ={sigma_ketchup})')
print('=' * 55)

# Soru 1: P(X < 35.8)
z_alt = (35.8 - mu_ketchup) / sigma_ketchup
p_alt = stats.norm.cdf(35.8, mu_ketchup, sigma_ketchup)
print(f'\nSoru 1: P(X < 35.8 oz)')
print(f'  Z = (35.8 - 36) / 0.11 = {z_alt:.2f}')
print(f'  P(X < 35.8) = {p_alt:.4f}  →  %{p_alt*100:.2f}')

# Soru 2: P(35.8 < X < 36.2) — kalite kontrol geçme
z_ust = (36.2 - mu_ketchup) / sigma_ketchup
p_ust = stats.norm.cdf(36.2, mu_ketchup, sigma_ketchup)
p_aralik = p_ust - p_alt
print(f'\nSoru 2: P(35.8 < X < 36.2) — kalite kontrolü geçme')
print(f'  Z_alt = {z_alt:.2f},  Z_üst = {z_ust:.2f}')
print(f'  P = {p_ust:.4f} - {p_alt:.4f} = {p_aralik:.4f}  →  %{p_aralik*100:.2f}')
print(f'\n  ✓ Doğru cevap: %{p_aralik*100:.2f}  (slayttaki %93.12 ile uyumlu)')

# ── Görsel ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.linspace(35.6, 36.4, 400)
y = stats.norm.pdf(x, mu_ketchup, sigma_ketchup)

def ciz_dagilim(ax, baslik, renk_aralik, alt=None, ust=None):
    ax.plot(x, y, color='black', lw=2)
    if alt is not None and ust is not None:
        mask = (x >= alt) & (x <= ust)
    elif alt is not None:
        mask = x <= alt
    elif ust is not None:
        mask = x >= ust
    ax.fill_between(x, y, where=mask, alpha=0.4, color=renk_aralik)
    ax.set_title(baslik, fontsize=11, fontweight='bold')
    ax.set_xlabel('oz'); ax.set_ylabel('Yoğunluk')
    ax.axvline(mu_ketchup, color='gray', ls=':', lw=1.5)

ciz_dagilim(axes[0], 'P(X < 35.8) — Az ketçap', MAVI, ust=35.8)
ciz_dagilim(axes[1], 'P(X < 36.2) — Sol alan', YESIL, ust=36.2)
ciz_dagilim(axes[2], 'P(35.8 < X < 36.2) — Kalite geçer', TURUNCU, alt=35.8, ust=36.2)

for ax in axes:
    ax.set_xticks([35.8, 36.0, 36.2])

plt.tight_layout()
plt.savefig('kalite_kontrol.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Eşik Noktası Bulma — Vücut Sıcaklığı (Slayt 15-16) ──────────────────────
mu_sicaklik = 98.2
sigma_sicaklik = 0.73

print('=' * 55)
print('Vücut Sıcaklığı: Eşik Noktası Hesaplama')
print(f'X ~ N(μ={mu_sicaklik}°F, σ={sigma_sicaklik}°F)')
print('=' * 55)

# En düşük %3 eşiği
z_03  = stats.norm.ppf(0.03)
x_03  = z_03 * sigma_sicaklik + mu_sicaklik
print(f'\nEn düşük %3 için eşik:')
print(f'  P(Z < z) = 0.03  →  z = {z_03:.3f}')
print(f'  x = z × σ + μ = {z_03:.3f} × {sigma_sicaklik} + {mu_sicaklik} = {x_03:.1f}°F')

# En yüksek %10 eşiği
z_90  = stats.norm.ppf(0.90)
x_90  = z_90 * sigma_sicaklik + mu_sicaklik
print(f'\nEn yüksek %10 için eşik:')
print(f'  P(Z < z) = 0.90  →  z = {z_90:.3f}')
print(f'  x = {z_90:.3f} × {sigma_sicaklik} + {mu_sicaklik} = {x_90:.1f}°F')
print(f'  ✓ Doğru cevap: 99.1°F (slayttaki (c) şıkkı)')

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_arr = np.linspace(95.5, 101, 400)
y_arr = stats.norm.pdf(x_arr, mu_sicaklik, sigma_sicaklik)

# Sol: en düşük %3
axes[0].plot(x_arr, y_arr, 'k-', lw=2)
axes[0].fill_between(x_arr, y_arr, where=(x_arr <= x_03), color=MAVI, alpha=0.4)
axes[0].axvline(x_03, color=MAVI, lw=2, ls='--')
axes[0].set_title(f'En düşük %3 → {x_03:.1f}°F', fontweight='bold')
axes[0].text(x_03 - 1.2, max(y_arr)*0.3, f'{x_03:.1f}°F', color=MAVI, fontweight='bold')
axes[0].set_xlabel('Sıcaklık (°F)')

# Sağ: en yüksek %10
axes[1].plot(x_arr, y_arr, 'k-', lw=2)
axes[1].fill_between(x_arr, y_arr, where=(x_arr >= x_90), color=TURUNCU, alpha=0.4)
axes[1].axvline(x_90, color=TURUNCU, lw=2, ls='--')
axes[1].set_title(f'En yüksek %10 → {x_90:.1f}°F', fontweight='bold')
axes[1].text(x_90 + 0.1, max(y_arr)*0.3, f'{x_90:.1f}°F', color=TURUNCU, fontweight='bold')
axes[1].set_xlabel('Sıcaklık (°F)')

plt.tight_layout()
plt.savefig('esik_noktasi.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 1.4 68 – 95 – 99.7 Kuralı (Ampirik Kural)

Normal dağılım için özet:

| Aralık | Yaklaşık Oran |
|--------|---------------|
| $\mu \pm 1\sigma$ | %68 |
| $\mu \pm 2\sigma$ | %95 |
| $\mu \pm 3\sigma$ | %99.7 |

In [ ]:
# ── 68-95-99.7 Kuralı Görselleştirmesi ───────────────────────────────────────
mu, sigma = 1500, 300  # SAT puanları
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 600)
y = stats.norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(13, 6))

# Gradyan dolgu
renkler = ['#d0e8f5', '#a3cfed', '#6aa6d3']
etiketler = ['μ±3σ → %99.7', 'μ±2σ → %95', 'μ±1σ → %68']

for k, (renk, etiket) in enumerate(zip(renkler, etiketler), 1):
    carp = 4 - k  # 3, 2, 1
    ax.fill_between(x, y,
                    where=((x >= mu - carp*sigma) & (x <= mu + carp*sigma)),
                    color=renk, alpha=0.9, label=etiket, zorder=k)

ax.plot(x, y, color='navy', lw=2.5, zorder=10)

# Dikey çizgiler ve etiketler
for k in range(-3, 4):
    val = mu + k*sigma
    ax.axvline(val, color='gray', lw=0.8, ls='--', alpha=0.7)
    ax.text(val, -0.00012, f'{val}\n({k:+d}σ)', ha='center', va='top', fontsize=9)

# Yatay ok ve yüzdeler
def cift_ok(ax, y_pos, xmin, xmax, yuzde, renk):
    ax.annotate('', xy=(xmax, y_pos), xytext=(xmin, y_pos),
                arrowprops=dict(arrowstyle='<->', color=renk, lw=2))
    ax.text((xmin+xmax)/2, y_pos + 0.00005, yuzde, ha='center', color=renk,
            fontweight='bold', fontsize=11)

cift_ok(ax, 0.00115, mu-sigma,   mu+sigma,   '%68',   '#1a6ba0')
cift_ok(ax, 0.00075, mu-2*sigma, mu+2*sigma, '%95',   '#155e8f')
cift_ok(ax, 0.00035, mu-3*sigma, mu+3*sigma, '%99.7', '#0e4d77')

ax.set_title('68-95-99.7 Ampirik Kuralı  (SAT: μ=1500, σ=300)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('SAT Puanı'); ax.set_ylabel('Yoğunluk')
ax.legend(loc='upper right', fontsize=11)
ax.set_ylim(-0.0003, None)

plt.tight_layout()
plt.savefig('ampirik_kural.png', dpi=150, bbox_inches='tight')
plt.show()

# Sayısal doğrulama
print('Sayısal doğrulama:')
for k in [1, 2, 3]:
    p = stats.norm.cdf(mu+k*sigma, mu, sigma) - stats.norm.cdf(mu-k*sigma, mu, sigma)
    print(f'  μ ± {k}σ  →  [{mu-k*sigma}, {mu+k*sigma}]  →  %{p*100:.2f}')

---
## 2. Geometrik Dağılım

### 2.1 Teori

Geometrik dağılım, **ilk başarıya kadar** kaç deneme yapıldığını modellemek için kullanılır.

**Koşullar:**
1. Denemeler **bağımsız** ve **özdeş dağılımlı** (b.o.d.)
2. Her denemede yalnızca iki sonuç: başarı (p) veya başarısızlık (1-p)

$$P(n.\text{ denemede ilk başarı}) = (1-p)^{n-1} \cdot p$$

| Parametre | Formül |
|-----------|--------|
| Ortalama | $\mu = \dfrac{1}{p}$ |
| Standart sapma | $\sigma = \sqrt{\dfrac{1-p}{p^2}}$ |

In [ ]:
# ── Milgram Deneyi — Geometrik Dağılım (Slayt 21-28) ─────────────────────────
p_reddet = 0.35  # şoku uygulamayı reddetme olasılığı

print('=' * 55)
print('Milgram Deneyi — Geometrik Dağılım')
print(f'p = {p_reddet} (şoku reddetme olasılığı)')
print('=' * 55)

for n in [1, 3, 10]:
    p_n = (1 - p_reddet)**(n-1) * p_reddet
    print(f'P({n}. denemede ilk başarı) = {(1-p_reddet)**(n-1):.4f} × {p_reddet} = {p_n:.4f}')

# Beklenen değer ve standart sapma
mu_geo  = 1 / p_reddet
std_geo = np.sqrt((1 - p_reddet) / p_reddet**2)
print(f'\nBeklenen değer : μ = 1/p = 1/{p_reddet} = {mu_geo:.2f} kişi')
print(f'Standart sapma : σ = {std_geo:.2f} kişi')

# Zar örneği (ekstra)
p_zar = 1/6
p_6_6 = (1 - p_zar)**5 * p_zar
print(f'\nEkstra örnek — Zarın 6. atışında ilk kez 6 gelme:')
print(f'P = (5/6)^5 × (1/6) = {p_6_6:.4f}')

# Görsel
n_vals = np.arange(1, 25)
p_vals = stats.geom.pmf(n_vals, p_reddet)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(n_vals, p_vals, color=MOR, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0].axvline(mu_geo, color=TURUNCU, lw=2.5, ls='--', label=f'μ={mu_geo:.2f}')
axes[0].set_title(f'Geometrik Dağılım (p={p_reddet})', fontweight='bold')
axes[0].set_xlabel('n. denemede ilk başarı (n)')
axes[0].set_ylabel('Olasılık')
axes[0].legend()

# Kümülatif
c_vals = np.cumsum(p_vals)
axes[1].step(n_vals, c_vals, color=MOR, lw=2.5, where='post')
axes[1].axhline(0.95, color=TURUNCU, lw=1.5, ls='--', label='%95 eşiği')
n95 = np.searchsorted(c_vals, 0.95) + 1
axes[1].axvline(n95, color=YESIL, lw=1.5, ls=':', label=f'n={n95}')
axes[1].set_title('Kümülatif Dağılım', fontweight='bold')
axes[1].set_xlabel('n'); axes[1].set_ylabel('P(X ≤ n)')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('geometrik_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n→ %95 olasılıkla ilk başarı en geç {n95}. denemede gerçekleşir.')

---
## 3. Binom Dağılımı

### 3.1 Teori

Binom dağılımı, **n sabit denemede tam olarak k başarı** elde etme olasılığını tanımlar.

**Koşullar:** Bağımsız denemeler, sabit n, iki sonuç (başarı/başarısızlık), sabit p.

$$P(\text{n denemede k başarı}) = \binom{n}{k} p^k (1-p)^{n-k}$$

| Parametre | Formül |
|-----------|--------|
| Ortalama | $\mu = np$ |
| Standart sapma | $\sigma = \sqrt{np(1-p)}$ |

**Kombinasyon formülü:**
$$\binom{n}{k} = \frac{n!}{k!(n-k)!}$$

In [ ]:
# ── Kombinasyon Fonksiyonu ────────────────────────────────────────────────────
from math import comb as kombinasyon

print('Kombinasyon örnekleri:')
print(f'  C(4,1) = {kombinasyon(4,1)}')
print(f'  C(9,2) = {kombinasyon(9,2)}')
print(f'  C(5,5) = {kombinasyon(5,5)}')
print(f'  C(5,0) = {kombinasyon(5,0)}')
print()

# Milgram: 4 kişiden tam 1'i reddeder
n, k, p = 4, 1, 0.35
p_manual = kombinasyon(n, k) * p**k * (1-p)**(n-k)
p_scipy  = stats.binom.pmf(k, n, p)
print(f'4 kişiden tam 1\'i şoku reddeder:')
print(f'  C(4,1) × 0.35^1 × 0.65^3 = {kombinasyon(4,1)} × {0.35:.4f} × {0.65**3:.4f}')
print(f'  = {p_manual:.4f}  (scipy: {p_scipy:.4f})')

In [ ]:
# ── Gallup Anketi — Obezite (Slayt 36-37, 40-42) ─────────────────────────────
p_obez = 0.262

print('=' * 55)
print('Gallup Anketi — Obezite Analizi')
print(f'p = {p_obez}  (Amerikalıların obez oranı)')
print('=' * 55)

# Soru 1: 10 kişiden tam 8'i obez
n1, k1 = 10, 8
p1 = stats.binom.pmf(k1, n1, p_obez)
print(f'\nP(10 kişiden tam 8\'i obez):')
print(f'  C(10,8) × {p_obez}^8 × {1-p_obez:.3f}^2 = {kombinasyon(10,8)} × ... = {p1:.5f}')
print(f'  → Oldukça düşük bir olasılık!')

# Soru 2: 100 kişilik örneklem
n2 = 100
mu_binom  = n2 * p_obez
std_binom = np.sqrt(n2 * p_obez * (1 - p_obez))
aralik_alt = mu_binom - 2 * std_binom
aralik_ust = mu_binom + 2 * std_binom

print(f'\n100 kişilik örneklem:')
print(f'  μ = np = {n2} × {p_obez} = {mu_binom:.1f} kişi')
print(f'  σ = √(np(1-p)) = {std_binom:.2f}')
print(f'  Olağan aralık: {mu_binom:.1f} ± 2×{std_binom:.2f} = ({aralik_alt:.1f}, {aralik_ust:.1f})')

# Ev eğitimi (slayt 43)
p_ev, n_ev = 0.13, 1000
mu_ev  = n_ev * p_ev
std_ev = np.sqrt(n_ev * p_ev * (1 - p_ev))
print(f'\nEv Eğitimi (%13, n=1000):')
print(f'  μ={mu_ev:.0f}, σ={std_ev:.1f}')
print(f'  Olağan aralık: ({mu_ev - 2*std_ev:.1f}, {mu_ev + 2*std_ev:.1f})')
print(f'  100 kişi bu aralıkta mı? → {"Hayır — alışılmadık!" if 100 < mu_ev - 2*std_ev else "Evet — olağan"}')

In [ ]:
# ── n arttıkça binom → normal yaklaşımı (Slayt 45-47) ────────────────────────
p_sabit = 0.10
n_degerler = [10, 30, 100, 300]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(n_degerler):
    k_vals = np.arange(0, n+1)
    binom_pmf = stats.binom.pmf(k_vals, n, p_sabit)
    
    # Normal yaklaşım
    mu_n  = n * p_sabit
    std_n = np.sqrt(n * p_sabit * (1 - p_sabit))
    x_n   = np.linspace(0, n, 400)
    
    axes[idx].bar(k_vals, binom_pmf, color=MAVI, alpha=0.7, label='Binom PMF',
                  width=0.8, edgecolor='white', lw=0.3)
    axes[idx].plot(x_n, stats.norm.pdf(x_n, mu_n, std_n), color=TURUNCU,
                   lw=2.5, label=f'N(μ={mu_n:.0f}, σ={std_n:.1f})')
    
    # Koşul kontrolü
    kosul = 'np≥10 ve n(1-p)≥10' if (n*p_sabit >= 10 and n*(1-p_sabit) >= 10) else 'Koşul sağlanmıyor'
    axes[idx].set_title(f'n={n}  ({kosul})', fontweight='bold')
    axes[idx].set_xlabel('Başarı sayısı (k)')
    axes[idx].set_ylabel('Olasılık')
    axes[idx].legend(fontsize=9)
    axes[idx].set_xlim(-1, max(20, n*p_sabit + 4*std_n))

plt.suptitle(f'Binom → Normal Yaklaşımı  (p={p_sabit})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('binom_normal.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ n büyüdükçe binom dağılımı normale yaklaşır.')

In [ ]:
# ── Binoma Normal Yaklaşım: Facebook Örneği (Slayt 49-51) ────────────────────
n_fb, p_fb = 245, 0.25
mu_fb  = n_fb * p_fb
std_fb = np.sqrt(n_fb * p_fb * (1 - p_fb))

print('=' * 55)
print('Facebook Güçlü Kullanıcı Analizi')
print(f'n={n_fb}, p={p_fb}')
print('=' * 55)
print(f'μ = {n_fb} × {p_fb} = {mu_fb:.2f}')
print(f'σ = √({n_fb}×{p_fb}×{1-p_fb}) = {std_fb:.2f}')

# P(X >= 70)
z_70   = (70 - mu_fb) / std_fb
p_norm = 1 - stats.norm.cdf(70, mu_fb, std_fb)
p_gercek = 1 - stats.binom.cdf(69, n_fb, p_fb)  # gerçek binom

print(f'\nP(X ≥ 70):')
print(f'  Z = (70 - {mu_fb:.2f}) / {std_fb:.2f} = {z_70:.2f}')
print(f'  Normal yaklaşım : {p_norm:.4f}  (%{p_norm*100:.2f})')
print(f'  Gerçek binom    : {p_gercek:.4f}  (%{p_gercek*100:.2f})')
print(f'  Fark            : {abs(p_norm-p_gercek):.5f}  (yaklaşım oldukça iyi!)')

# Görsel
k_vals = np.arange(20, 110)
binom_pmf = stats.binom.pmf(k_vals, n_fb, p_fb)
x_norm = np.linspace(20, 110, 400)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(k_vals, binom_pmf, width=0.8, color=MAVI, alpha=0.6, label=f'Bin({n_fb},{p_fb})',
       edgecolor='white', lw=0.3)
ax.bar(k_vals[k_vals >= 70], binom_pmf[k_vals >= 70], width=0.8,
       color=TURUNCU, alpha=0.8, label=f'P(X≥70) = {p_gercek:.3f}')
ax.plot(x_norm, stats.norm.pdf(x_norm, mu_fb, std_fb), color='navy',
        lw=2.5, label=f'N({mu_fb:.1f},{std_fb:.2f})')
ax.axvline(70, color=TURUNCU, lw=2, ls='--')
ax.set_title('Facebook: Güçlü Kullanıcı Olasılığı (Binom ≈ Normal)', fontweight='bold')
ax.set_xlabel('Güçlü kullanıcı sayısı (k)')
ax.set_ylabel('Olasılık')
ax.legend()

plt.tight_layout()
plt.savefig('facebook_binom.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Negatif Binom Dağılımı

### 4.1 Teori

Negatif binom dağılımı, **n. denemede k. başarıyı** gözlemleme olasılığını tanımlar.

$$P(n.\text{ denemede }k.\text{ başarı}) = \binom{n-1}{k-1} p^k (1-p)^{n-k}$$

**Binom vs Negatif Binom:**
| | Binom | Negatif Binom |
|-|-------|---------------|
| Sabit olan | n (deneme sayısı) | k (başarı sayısı) |
| İstenen | k başarı | n. deneme |
| Son deneme | Herhangi | **Mutlaka başarı** |

In [ ]:
# ── Psikoloji Lab Örneği — Negatif Binom (Slayt 54) ──────────────────────────
p_cift = 0.10   # çift bulma olasılığı
k_hedef = 10    # hedef: 10 çift
n_sorgu = 30    # 30. denemede 10. başarı

# Formül: C(n-1, k-1) * p^k * (1-p)^(n-k)
p_negbinom = kombinasyon(n_sorgu-1, k_hedef-1) * p_cift**k_hedef * (1-p_cift)**(n_sorgu-k_hedef)
p_scipy_nb = stats.nbinom.pmf(n_sorgu - k_hedef, k_hedef, p_cift)

print('=' * 55)
print('Psikoloji Laboratuvarı — Negatif Binom')
print(f'p={p_cift}, k={k_hedef}, n={n_sorgu}')
print('=' * 55)
print(f'\nP(30. denemede 10. başarı):')
print(f'  C(29,9) × {p_cift}^10 × {1-p_cift:.1f}^20')
print(f'  = {kombinasyon(29,9):,} × {p_cift**10:.2e} × {(0.9)**20:.4f}')
print(f'  = {p_negbinom:.5f}')
print(f'  (scipy: {p_scipy_nb:.5f})')

# Dağılımın görselleştirilmesi
n_vals = np.arange(k_hedef, 80)
pmf_vals = stats.nbinom.pmf(n_vals - k_hedef, k_hedef, p_cift)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(n_vals, pmf_vals, color=YESIL, alpha=0.8, edgecolor='white', lw=0.3,
       label=f'k={k_hedef} başarı için deneme sayısı')
ax.bar([30], [p_negbinom], color=TURUNCU, alpha=1.0, edgecolor='black',
       label=f'n=30: P={p_negbinom:.5f}', width=0.8)

mu_nb = k_hedef / p_cift
ax.axvline(mu_nb, color='red', lw=2, ls='--', label=f'μ = k/p = {mu_nb:.0f}')
ax.set_title(f'Negatif Binom Dağılımı  (k={k_hedef}, p={p_cift})', fontweight='bold')
ax.set_xlabel('Toplam deneme sayısı (n)')
ax.set_ylabel('Olasılık')
ax.legend()

plt.tight_layout()
plt.savefig('negatif_binom.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n→ Beklenen deneme sayısı: μ = k/p = {k_hedef}/{p_cift} = {mu_nb:.0f}')

---
## 5. Poisson Dağılımı

### 5.1 Teori

Poisson dağılımı, **nadir olayların** belirli bir zaman/alan biriminde kaç kez gerçekleşeceğini modellemek için kullanılır.

**Kullanım koşulları:**
- Olaylar nadir
- Nüfus büyük
- Olaylar birbirinden bağımsız

$$P(\text{k olay}) = \frac{\lambda^k e^{-\lambda}}{k!}$$

- $\lambda$: birim zamandaki ortalama olay sayısı  
- $\mu = \sigma^2 = \lambda$

In [ ]:
# ── Elektrik Kesintisi Örneği (Slayt 58-59) ───────────────────────────────────
lambda_haftalik = 2  # haftada ortalama 2 kesinti

print('=' * 55)
print('Elektrik Kesintisi — Poisson Dağılımı')
print(f'λ (haftalık) = {lambda_haftalik}')
print('=' * 55)

# Soru 1: Bir haftada tam 1 kesinti
k1 = 1
p_hafta_1 = stats.poisson.pmf(k1, lambda_haftalik)
el = np.e
print(f'\nP(haftada 1 kesinti):')
print(f'  λ^k × e^(-λ) / k! = {lambda_haftalik}^1 × e^(-{lambda_haftalik}) / 1!')
print(f'  = {lambda_haftalik} × {np.exp(-lambda_haftalik):.4f} / 1 = {p_hafta_1:.4f}')

# Soru 2: Belirli bir günde 3 kesinti
lambda_gunluk = lambda_haftalik / 7
k2 = 3
p_gun_3 = stats.poisson.pmf(k2, lambda_gunluk)
print(f'\nP(günde 3 kesinti):')
print(f'  λ_gün = {lambda_haftalik}/7 = {lambda_gunluk:.4f}')
print(f'  P = {lambda_gunluk:.4f}^3 × e^(-{lambda_gunluk:.4f}) / 3! = {p_gun_3:.6f}')

# Görsel: farklı λ değerleri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
k_range = np.arange(0, 15)

for lam, color, label in [(1, MAVI, 'λ=1'), (2, TURUNCU, 'λ=2'),
                           (5, YESIL, 'λ=5'), (10, MOR, 'λ=10')]:
    axes[0].plot(k_range, stats.poisson.pmf(k_range, lam), 'o-',
                 color=color, lw=2, ms=6, label=label)
axes[0].set_title('Poisson PMF — Farklı λ değerleri', fontweight='bold')
axes[0].set_xlabel('k (olay sayısı)')
axes[0].set_ylabel('Olasılık')
axes[0].legend()

# λ=2 için çubuk
pmf2 = stats.poisson.pmf(k_range, lambda_haftalik)
renkler_bar = [TURUNCU if k == 1 else MAVI for k in k_range]
axes[1].bar(k_range, pmf2, color=renkler_bar, alpha=0.8, edgecolor='white')
axes[1].set_title(f'Poisson(λ={lambda_haftalik}) — Haftada kesinti sayısı', fontweight='bold')
axes[1].set_xlabel('k (kesinti sayısı)')
axes[1].set_ylabel('Olasılık')
axes[1].text(1, pmf2[1] + 0.005, f'P(k=1)\n={pmf2[1]:.3f}',
             ha='center', color=TURUNCU, fontweight='bold')

plt.tight_layout()
plt.savefig('poisson_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Dağılımların Karşılaştırmalı Özeti

In [ ]:
# ── Tüm Dağılımları Bir Arada Karşılaştır ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 1. Normal
x = np.linspace(-4, 4, 300)
axes[0].plot(x, stats.norm.pdf(x), color=MAVI, lw=2.5)
axes[0].fill_between(x, stats.norm.pdf(x), alpha=0.15, color=MAVI)
axes[0].set_title('Normal  N(0,1)', fontweight='bold', color=MAVI)
axes[0].set_xlabel('x')

# 2. Geometrik
n_geo = np.arange(1, 20)
axes[1].bar(n_geo, stats.geom.pmf(n_geo, 0.35), color=MOR, alpha=0.8, edgecolor='white')
axes[1].set_title('Geometrik  (p=0.35)', fontweight='bold', color=MOR)
axes[1].set_xlabel('n. denemede ilk başarı')

# 3. Binom
k_b = np.arange(0, 21)
axes[2].bar(k_b, stats.binom.pmf(k_b, 20, 0.35), color=YESIL, alpha=0.8, edgecolor='white')
axes[2].set_title('Binom  Bin(20, 0.35)', fontweight='bold', color=YESIL)
axes[2].set_xlabel('k başarı')

# 4. Negatif Binom
n_nb = np.arange(3, 30)
axes[3].bar(n_nb, stats.nbinom.pmf(n_nb - 3, 3, 0.35), color=TURUNCU, alpha=0.8, edgecolor='white')
axes[3].set_title('Negatif Binom  (k=3, p=0.35)', fontweight='bold', color=TURUNCU)
axes[3].set_xlabel('n. denemede 3. başarı')

# 5. Poisson
k_p = np.arange(0, 15)
axes[4].bar(k_p, stats.poisson.pmf(k_p, 3), color='#E91E63', alpha=0.8, edgecolor='white')
axes[4].set_title('Poisson  (λ=3)', fontweight='bold', color='#E91E63')
axes[4].set_xlabel('k olay')

# 6. Özet tablo
axes[5].axis('off')
tablo_veri = [
    ['Dağılım', 'Tür', 'Parametre', 'μ', 'σ'],
    ['Normal', 'Sürekli', 'μ, σ', 'μ', 'σ'],
    ['Geometrik', 'Kesikli', 'p', '1/p', '√((1-p)/p²)'],
    ['Binom', 'Kesikli', 'n, p', 'np', '√(np(1-p))'],
    ['Neg. Binom', 'Kesikli', 'k, p', 'k/p', '√(k(1-p)/p²)'],
    ['Poisson', 'Kesikli', 'λ', 'λ', '√λ'],
]
tablo = axes[5].table(cellText=tablo_veri[1:], colLabels=tablo_veri[0],
                       loc='center', cellLoc='center')
tablo.auto_set_font_size(False)
tablo.set_fontsize(10)
tablo.scale(1.2, 2.0)
for (r, c), cell in tablo.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2E86AB')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f0f4f8')
axes[5].set_title('Dağılımlar Özet Tablosu', fontweight='bold', pad=20)

plt.suptitle('Temel Olasılık Dağılımları — Karşılaştırma', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('dagilimlar_ozet.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Alıştırmalar

Aşağıdaki alıştırmaları Python ile çözünüz.

In [ ]:
# ── Alıştırma 1: Z Skoru ──────────────────────────────────────────────────────
# Bir öğrencinin matematik sınavı puanı 78'dir.
# Sınıf ortalaması 70, standart sapması 8'dir.
# Bu öğrencinin Z skoru nedir? Kaçıncı yüzdelikte yer almaktadır?

puan, mu_sinav, sigma_sinav = 78, 70, 8

# BURAYA KOD YAZIN
# z = ...
# yuzdelik = ...
# print(...)

In [ ]:
# ── Alıştırma 2: Binom ────────────────────────────────────────────────────────
# Bir sınavda her sorunun 4 şıkkı var ve öğrenci rastgele cevaplıyor.
# 20 sorudan tam 5'ini doğru yapma olasılığı nedir?
# Kaç soru doğru yapması beklenir?

n_soru = 20
p_dogru = 1/4

# BURAYA KOD YAZIN
# p_5 = ...
# beklenen = ...
# print(...)

In [ ]:
# ── Alıştırma 3: Poisson ──────────────────────────────────────────────────────
# Bir çağrı merkezine saatte ortalama 15 çağrı geliyor.
# Herhangi bir dakikada tam 2 çağrı gelme olasılığı nedir?

lambda_saat = 15
lambda_dakika = lambda_saat / 60

# BURAYA KOD YAZIN
# p_2 = ...
# print(...)

In [ ]:
# ── Alıştırma Cevapları ───────────────────────────────────────────────────────
print('CEVAPLAR')
print('='*50)

# 1
puan, mu_sinav, sigma_sinav = 78, 70, 8
z = (puan - mu_sinav) / sigma_sinav
yuzdelik = stats.norm.cdf(z) * 100
print(f'1) Z = {z:.2f}, Yüzdelik = {yuzdelik:.1f}. yüzdelik')

# 2
n_soru, p_dogru = 20, 1/4
p_5 = stats.binom.pmf(5, n_soru, p_dogru)
beklenen = n_soru * p_dogru
print(f'2) P(X=5) = {p_5:.4f}, Beklenen = {beklenen:.1f}')

# 3
lambda_d = 15/60
p_2 = stats.poisson.pmf(2, lambda_d)
print(f'3) λ_dakika = {lambda_d:.4f}, P(X=2) = {p_2:.4f}')

---
## 📋 Bölüm Özeti

| Dağılım | Ne Zaman Kullanılır? | Temel Fonksiyon |
|---------|----------------------|-----------------|
| **Normal** | Sürekli veriler, çan eğrisi | `stats.norm.cdf/ppf` |
| **Geometrik** | İlk başarıya kadar deneme sayısı | `stats.geom.pmf` |
| **Binom** | n denemede k başarı (sabit n) | `stats.binom.pmf` |
| **Negatif Binom** | k. başarı için n deneme (sabit k) | `stats.nbinom.pmf` |
| **Poisson** | Nadir olaylar, birim zamanda sayım | `stats.poisson.pmf` |

**Önemli bağlantılar:**
- Binom → Normal: $np \geq 10$ ve $n(1-p) \geq 10$ koşullarında
- Binom → Poisson: $n \to \infty$, $p \to 0$, $np = \lambda$ sabit kalırken
- Geometrik, Negatif Binom'un özel halidir ($k=1$)

---
*Haydar Kılıç — Yapay Zeka Mühendisliği*